In [1]:
from vehicle_kg import TYPE_TO_DOMAIN, VALID_CONNECTIONS
import re, textwrap, torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from tqdm import tqdm
from transformers import TrainerCallback, EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import pandas as pd


from typing import List, Dict, Tuple, Set, Optional

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
OUTPUT_DIR = "./lora-output"

# LoRA settings
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training settings
EPOCHS = 10
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [2]:
def get_kg_context(code: str) -> str:
    """Scans code and retrieves relevant domain rules."""

    found_domains: Set[str] = set()
    context_lines: List[str] = []
    
    # Find all matching types and their domains
    for type_name, domain in TYPE_TO_DOMAIN.items():
        if re.search(rf'\b{re.escape(type_name)}\b', code):
            context_lines.append(f"- '{type_name}' belongs to Domain: {domain}")
            found_domains.add(domain)
    
    # Add connection rules if domains were found
    if found_domains:
        context_lines.append("\nValid Connections Rules:")
        for domain in sorted(found_domains):  # Sort for consistency
            allowed = VALID_CONNECTIONS.get(domain, [])
            context_lines.append(f"- {domain} can ONLY connect to: {allowed}")
    
    return "\n".join(context_lines)


def parse_error_message(error_string: str) -> Tuple[str, str]:
    """Parses an error message into error type and description."""

    cleaned = error_string.replace("ERROR:", "").strip()
    
    if " (line : " in cleaned:
        cleaned = cleaned.split(" (line : ")[0].strip()
    
    parts = cleaned.split(":", 1)
    error_type = parts[0].strip()
    message = parts[1].strip() if len(parts) > 1 else ""
    
    return error_type, message


SOLUTION_MAP = {
    "Type mismatch": "To solve this, the attribute declaration should be renamed correctly based on semantic meaning",
    "Domain violation": "To solve this, we need to reroute this connection to a compatible port as per the rules",
    "Quantity mismatch": "To solve this, we need to assign correct units to this attribute",
    "Unit expression corruption": "To solve this, the unit must be corrected to valid form"
}


def create_thought(error_message: str, mutation_category: str) -> str:
    """Generates analysis thought process based on error type."""
    
    if mutation_category == "domain":
        error_type, error_desc = parse_error_message(error_message)
        solution = SOLUTION_MAP.get(error_type, "")
        return (
            f"Let's think step-by-step.\n"
            f"Checking the rules, reading the code.\n"
            f"{error_desc}.\n"
            f"{solution}."
        )
    elif mutation_category == "syntax":
        return (
            "Let's think step-by-step.\n"
            "The compiler reports syntax errors.\n"
            "To solve this, we need to fix the syntax issues in the code at the reported lines."
        )
    
    return ""


def create_fix(patches: List[Dict[str, str]]) -> str:
    """Formats patch instructions into a readable fix description."""
    
    fixes: List[str] = []
    
    for i, patch in enumerate(patches):
        context = patch.get("context", "")
        before = patch.get("before", "")
        after = patch.get("after", "")
        
        parts: List[str] = []
        
        # Add separator for subsequent patches
        parts.append("\n\n## AND " if i > 0 else "### ")
        
        # Add context if present
        if context:
            parts.append(f"AFTER THIS CODE:\n{context}\n\n")
        
        # Determine and format operation type
        if before and after:
            parts.append(f"## REPLACE:\n{before}\n\n## WITH:\n{after}")
        elif before:
            parts.append(f"## DELETE:\n{before}")
        elif after:
            parts.append(f"## INSERT:\n{after}")
        
        fixes.append("".join(parts))
    
    return "".join(fixes)


def create_prompt(error_message: str, mutation_category: str, bad_code: str) -> str:
    """Creates a prompt for the LLM based on error type."""

    if mutation_category == "syntax":
        return textwrap.dedent(f"""\
            Analyze the following SysML v2 code for errors reported by the compiler. 
            Think step by step, and then provide fixes.
            
            ### Compiler Error:
            {error_message}
            
            ### Code:
            {bad_code}
        """)
    
    elif mutation_category == "domain":
        rules = get_kg_context(bad_code)
        return textwrap.dedent(f"""\
            Analyze the SysML v2 code for potential semantic domain inconsistencies using the provided relevant rules. 
            Think step by step, and then give fixes if any.
            
            ### Domain Rules:
            {rules}
            
            ### Code:
            {bad_code}
        """)
    
    return ""

def create_completion(error_message: str, mutation_category: str, patches: List[Dict[str, str]]) -> str:
    """Creates the completion response containing analysis and fix."""

    thought = create_thought(error_message, mutation_category)
    fix = create_fix(patches)

    return f"### ANALYSIS:\n{thought}\n\n### FIX:\n{fix}"


def processing_function(examples):
    texts, lengths = [], []
    
    for err, cat, code, patch in zip(
        examples["error_message"],
        examples["mutation_category"],
        examples["bad_code"],
        examples["fix_patches"]
    ):
        
        question = create_prompt(err, cat, code)
        answer = create_completion(err, cat, patch) + tokenizer.eos_token

        chat = [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ]

        text = tokenizer.apply_chat_template( # here only done for computing lengths, internally done automatically so just pass dict chats
            chat,
            tokenize=False,
            add_generation_prompt=False
        )

        texts.append(chat)
        lengths.append(len(tokenizer(text)["input_ids"]))
       
    return {"messages": texts, "length": lengths}

In [3]:
df = pd.read_json("complete_sysmlv2_dataset.jsonl", lines=True)
ds = Dataset.from_pandas(df).map(
    processing_function, 
    batched=True
)
MAX_LEN = 2048
ds = ds.filter(lambda x: x["length"] <= MAX_LEN)

Map:   0%|          | 0/6899 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6899 [00:00<?, ? examples/s]

In [ ]:
from split_dataset import split_dataset

train_ds, val_ds, test_ds = split_dataset(ds)

Split sizes: train=5356, val=972, test=975


In [5]:
# Load model with 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda:0"
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)


peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="messages",
    max_length=2048,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,  # Add this for validation
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    #assistant_only_loss=True,

    optim="paged_adamw_8bit",
    packing=False,
    
    # Validation & early stopping
    eval_strategy="steps",              
    save_strategy="steps",               
    load_best_model_at_end=True,        
    metric_for_best_model="eval_loss", 
    greater_is_better=False,            
    
    logging_steps=10,
    fp16=True,
)

trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=training_args,
        peft_config=peft_config
    )

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


c:\Data\Research\Doctoral\INCOSE\code\incose\.venv\Lib\site-packages\peft\tuners\lora\bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
c:\Data\Research\Doctoral\INCOSE\code\incose\.venv\Lib\site-packages\peft\tuners\tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/5356 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5356 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/972 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/972 [00:00<?, ? examples/s]

In [6]:
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f"\nTraining complete! RAG-Model saved to {OUTPUT_DIR}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
c:\Data\Research\Doctoral\INCOSE\code\incose\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,2.309700,1.825931,1.879011,16839.000000,0.649866
20,1.550200,1.141581,1.297762,34302.000000,0.749938
30,0.907000,0.729182,0.690181,51392.000000,0.852753
40,0.700200,0.601118,0.582386,68426.000000,0.864825
50,0.594100,0.522780,0.503999,85229.000000,0.879807
60,0.442800,0.486384,0.491243,102268.000000,0.885225
70,0.425300,0.450879,0.445643,119792.000000,0.889551
80,0.365200,0.423387,0.441419,136296.000000,0.894013
90,0.474500,0.416119,0.417088,153680.000000,0.893842
100,0.440500,0.404527,0.400456,171738.000000,0.896556


c:\Data\Research\Doctoral\INCOSE\code\incose\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Data\Research\Doctoral\INCOSE\code\incose\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two


Training complete! RAG-Model saved to ./lora-output
